# 📘 Azure Databricks Storage & Unity Catalog Notes

---

## 🔹 Managed Resource Group Storage (Important)

When an Azure Databricks workspace is created, it automatically creates:
- A **managed resource group**
- A **storage account (ADLS Gen2)**

### 📦 Containers created internally:
- `root`
- `meta`
- `logs`
- `jobs`
- `ephemeral`
- `unity-catalog-storage`

### 🚫 Key Limitation:
- This storage is **Databricks-managed**
- Protected by **Deny Assignment**
- ❌ Cannot be accessed directly
- ❌ Cannot be used for:
  - Unity Catalog
  - External tables
  - Mount points

👉 This storage is **only for internal Databricks use**

---

## 🔹 DBFS (Databricks File System)

Command:
```python
display(dbutils.fs.ls("/"))
```
---

# 📘 Azure Databricks Storage & Unity Catalog Notes

## 📁 DBFS (Databricks File System)

### 🔹 What you see:

```python
/databricks-datasets/
/databricks-results/
/Volumes/
```

---

### 🧠 Important:
- DBFS is a **virtual abstraction layer**
- It does **NOT show actual ADLS containers**
- Internal containers like `root`, `meta`, `logs` are **hidden**
- It is only a **logical filesystem view inside Databricks**

---

## 🔹 Unity Catalog Basics

### 🏗️ Object Hierarchy:

```python
Catalog
└── Schema (Database)
└── Tables / Views
```

### 📌 Example:
sales_catalog.europe.customers

---

## 🔹 Creating a Catalog

### ❌ Without storage (may fail):
```sql
CREATE CATALOG my_catalog;
```

Error: **Metastore storage root URL does not exist**

```python
AnalysisException: [RequestId=c01f0bc2-bf67-40de-9eb3-7da4ffc6e3e3 ErrorClass=INVALID_STATE] Metastore storage root URL does not exist. Default Storage is enabled in your account. 
You can use the UI to create a new catalog using Default Storage, or please provide a storage location for the catalog (for example 'CREATE CATALOG myCatalog MANAGED LOCATION '<location-path>')
```

🔍 Why this error occurs
- Unity Catalog metastore has no default storage configured
- Databricks does not know where to physically store data
- So catalog creation fails without storage info

✅ With Managed Location (Correct approach)

```sql
CREATE CATALOG my_catalog
MANAGED LOCATION 'abfss://<container>@<storage-account>.dfs.core.windows.net/';
```

# 🏗️ Recommended Setup (Best Practice)

## ✔️ Step 1: Create your own ADLS Gen2 storage
- Create Storage Account in Azure Data Lake Storage Gen2  
- Create Container (example: `uc-data`)

---

## ✔️ Step 2: Grant access to :contentReference[oaicite:1]{index=1}

Use one of the following authentication methods:

- **Managed Identity (Recommended)**
- **Service Principal**

---

## ✔️ Step 3: Create Unity Catalog properly

```sql
CREATE CATALOG my_catalog
MANAGED LOCATION 'abfss://uc-data@myadls.dfs.core.windows.net/';

# 🚫 Important Clarification (Very Important)

## ❌ Do NOT use Databricks-managed storage:

- root  
- meta  
- logs  
- jobs  
- unity-catalog-storage  

---

## 🧠 Why:

- It is system-controlled  
- Protected by Deny Assignment  
- Not accessible for user workloads  
- Not supported for Unity Catalog usage  

---

# 📊 Quick Summary Table

| Component | Purpose | Access |
|----------|--------|--------|
| Managed ADLS (Databricks RG) | Internal system storage | ❌ No access |
| DBFS | Virtual filesystem view | ✅ Limited access |
| Your ADLS Gen2 | Data storage | ✅ Full control |
| Unity Catalog | Governance layer | ✅ Recommended |

# 🧭 Optional Advanced Learning

If you want to go deeper, you can explore:

- External Locations in Unity Catalog
- Storage Credentials (IAM-based access)
- Managed vs External Tables
- Unity Catalog Permissions model
- Data lineage tracking

# Unity Catalog in Databricks – Complete Guide

## 1. What is Unity Catalog?
Unity Catalog is a **centralized data governance solution** in Databricks that manages:
- Data access permissions
- Data discovery
- Data lineage
- Data auditing
- Data sharing

It provides **fine-grained access control** for all data assets in Databricks like:
- Tables
- Views
- Files
- Volumes
- Models
- Functions

Unity Catalog works across **all workspaces** and provides a **single governance layer**.

## 2. What is Metastore?

**The metastore is the top-level container for metadata in Unity Catalog. It registers metadata about data and the permissions that govern access to them. Foa a workspace to use Unity Catalog, it must have a Unity Catalog metastore attached. You should have one metastore for each region in which you have workspaces.**<p>
Unlike Hivde metastore, the Unity Catalog metastore is not a service boundary: it runs in a multi-tenant environment and represents a logical boundary for the segregation of data by region for a given Databricks account.

![image_1774386461977.png](./Images/image_1774386461977.png "image_1774386461977.png")

**DataBricks Architecture**
![image_1774386482610.png](./Images/image_1774386482610.png "image_1774386482610.png")

# 3. Why Unity Catalog?
Before Unity Catalog:
- Each workspace had separate metastore
- Access control was at cluster level
- Hard to manage permissions
- No proper data lineage
- No centralized governance

With Unity Catalog:
- Central governance
- Row level security
- Column level security
- Data lineage
- Data audit logs
- Data sharing
- Managed tables & external tables
- Better security model

---

# 4. Unity Catalog Architecture
Unity Catalog has a **3-level namespace**:


# Managed storage location hierarchy

The level at which you define managed storage in Unity Catalog depends on your preferred data isolation model. Your organization may require that certain types of data be stored within specific accounts or buckets in your cloud tenant.

Unity Catalog gives you the ability to configure managed storage locations at the metastore, catalog, or schema level to satisfy such requirements.

**For example, let's say your organization has a company compliance policy that requires production data relating to human resources to reside in the bucket s3://mycompany-hr-prod. In Unity Catalog, you can achieve this requirement by setting a location on a catalog level, creating a catalog called, for example hr_prod, and assigning the location s3://mycompany-hr-prod/unity-catalog to it. This means that managed tables or volumes created in the hr_prod catalog (for example, using CREATE TABLE hr_prod.default.table …) store their data in s3://mycompany-hr-prod/unity-catalog. Optionally, you can choose to provide schema-level locations to organize data within the hr_prod catalog at a more granular level.**

If storage isolation is not required for some catalogs, you can optionally set a storage location at the metastore level. This location serves as a default location for managed tables and volumes in catalogs and schemas that don't have assigned storage. Typically, however, Databricks recommends that you assign separate managed storage locations for each catalog.

The system evaluates the hierarchy of storage locations from schema to catalog to metastore.

For example, if a table myCatalog.mySchema.myTable is created in my-region-metastore, the table storage location is determined according to the following rule:

- If a location has been provided for mySchema, it will be stored there.
- If not, and a location has been provided on myCatalog, it will be stored there.
- Finally, if no location has been provided on myCatalog, it will be stored in the location associated with the my-region-metastore.

![image_1774460993944.png](./Images/image_1774460993944.png "image_1774460993944.png")

# Metastore Level Storage
- Metastore is attached to the Data Lake
- Schema is managed
- Object is managed


Now, data for our objects[managed tables] will be saved in the Metastore Data Lake

In [0]:
%sql
SELECT current_catalog();

In [0]:
spark

In [0]:
display(dbutils.fs.ls("/"))

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS managed_catalog_02;
-- MANAGED LOCATION '<location path>'
-- COMMENT "Managed metastore location";

-- CREATE SCHEMA IF NOT EXISTS managed_catalog.managed_schema
-- COMMENT <comment>; 


-- CREATE TABLE IF NOT EXISTS managed_catalog.managed_schema.managed_table (
--   id INT,
--   name STRING,
--   value DOUBLE
-- )
-- COMMENT <comment>;

-- INSERT INTO managed_catalog.managed_schema.managed_table
-- VALUES (1, 'A', 1.0),
--        (2, 'B', 2.0),
--        (3, 'C', 3.0);

-- USE managed_catalog.managed_schema;
-- DESCRIBE TABLE managed_table;
-- SELECT * FROM managed_catalog.managed_schema.managed_table;
-- DROP TABLE IF EXISTS managed_catalog.managed_schema.managed_table;
-- DROP SCHEMA IF EXISTS managed_catalog.managed_schema;
-- DROP CATALOG IF EXISTS managed_catalog;




# Metastore Level Storage
- Metastore is attached to the Data Lake
- Catalog is also attached to the Data Lake
- Schema is managed
- Object is managed


Now, data for our objects[managed tables] will be saved in the Metastore Data Lake

# Mount external storage (Best practice)

Instead of using the managed storage, create your own storage account and mount it:
Example (ADLS Gen2):

```python

configs = {
  "fs.azure.account.key.<storage-account>.dfs.core.windows.net": "<access-key>"
}

dbutils.fs.mount(
  source = "abfss://<container>@<storage-account>.dfs.core.windows.net/",
  mount_point = "/mnt/mydata",
  extra_configs = configs)
  
```

# Use Unity Catalog (Modern approach)

If you're using newer Databricks setups:

- Create external locations
- Assign permissions via Unity Catalog
- Access data securely without mounts